# Механические напряжения по сдвигу фонона

* **Si:** σ (МПа) = −435 · Δω (см⁻¹), De Wolf (1996, 2003). Пик выше эталона — сжатие.
* **GaAs:** пока только оценка изменения объёма ΔV/V через параметр Грюнайзена
  (γ LO = 1.23, TO = 1.39; Wickboldt et al. 1987). Для двуосного напряжения нужны
  потенциалы p, q (Cerdeira et al. 1972) — их ещё не сверили.

Главное правило: Δω считается от **ненапряжённого эталона, снятого в тот же день на том же
лазере**. Иначе сдвиг калибровки прибора (десятые доли см⁻¹) выглядит как напряжение
в десятки–сотни МПа.

Нажмите **Run All**. Файлы `../data/raw/own/*.txt` появляются после `scripts/import_own.py`.

In [ ]:
import pandas as pd
from ramancore.io import read_spectrum
from ramancore.peaks import fit_peaks
from ramancore.semiconductors import phonon_quality, si_stress, volume_change

## 1. Si

Подставьте спектр своей структуры (`my_file`) и ненапряжённой пластины Si (`ref_file`).
Ниже для примера два калибровочных спектра на разных лазерах: разница 0.3 см⁻¹ —
это не напряжение, а разная калибровка 532 и 785 нм, ~120 МПа «из ничего».

In [ ]:
ref_file = "../data/raw/own/own_si_cal_532.txt"  # ← ненапряжённый Si
my_file = "../data/raw/own/own_si_cal_785.txt"   # ← ваш образец

def si_center(path):
    x, y, _ = read_spectrum(path)
    table, _ = fit_peaks(x, y, [520.7], shape="voigt", margin=20)
    return table.center[0], table.center_err[0]

w0, _ = si_center(ref_file)
w, w_err = si_center(my_file)
sigma, sigma_err = si_stress(w, w0, w_err)
print(f"Эталон {w0:.2f}, образец {w:.2f} см⁻¹  →  σ = {sigma:.0f} ± {sigma_err:.0f} МПа",
      "(сжатие)" if sigma < 0 else "(растяжение)")

## 2. GaAs: напряжение или нет?

При всестороннем сжатии TO и LO сдвигаются **в одну сторону** и дают одинаковый ΔV/V.
Если знаки разные — сдвиг не от напряжения (легирование, дефекты, калибровка).

In [ ]:
x, y, _ = read_spectrum("../data/raw/own/own_gaas_vs182_294K.txt")
ref = phonon_quality(x, y, "GaAs")["table"]
rows = []
for name in ["te_A", "te_B", "te_V", "te_G", "te_D", "te_E", "ar97", "ar154s", "ar889", "ar104", "t771"]:
    x, y, _ = read_spectrum(f"../data/raw/own/own_gaas_{name}.txt")
    t = phonon_quality(x, y, "GaAs")["table"]
    dv_lo, _ = volume_change(t.loc["LO", "center"], ref.loc["LO", "center"], "GaAs", "LO")
    dv_to, _ = volume_change(t.loc["TO", "center"], ref.loc["TO", "center"], "GaAs", "TO")
    rows.append({"образец": name, "LO": t.loc["LO", "center"], "TO": t.loc["TO", "center"],
                 "ΔV/V по LO, %": dv_lo, "ΔV/V по TO, %": dv_to,
                 "похоже на напряжение": bool(dv_lo * dv_to > 0)})
pd.DataFrame(rows).round(2)